In [1]:
!pip install -q gradio transformers ftfy duckduckgo-search
!pip install -q git+https://github.com/openai/CLIP.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 39.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
"""
Few-Shot Learning with CLIP, BLIP, and DINOv2 -- Interactive Demo
IUT CSE Semester 6 -- CVLab

Every tab in this notebook runs live, on whatever you upload, using publicly
available pretrained backbones (CLIP, BLIP, DINOv2). No hardcoded results
tables, no lookups.

IMPORTANT -- what "live" means for each method in the K-Shot Lab below:
  - CLIP zero-shot / nearest-prototype, DINOv2 nearest-prototype, and
    Tip-Adapter are training-free: every number is a genuine forward pass +
    closed-form computation on YOUR uploaded images, right now.
  - "CLIP-LoRA (live probe)" is a lightweight linear adapter trained for a
    handful of gradient steps on your support embeddings, in the same spirit
    as CLIP-LoRA (rapid few-shot adaptation from very few examples). It is
    NOT the paper's exact setup (low-rank adapters inserted into CLIP's
    attention layers, trained end-to-end) -- that needs backprop through the
    encoder and trained checkpoints we don't have here. Say so if a professor
    asks: this is a live approximation of the idea, not a reproduction of the
    report's numbers.
  - 2SFS here is prototype-stage-1 + a calibrated linear stage-2, also
    trained live in seconds on your support set.
  - MaPLe is left out: it needs joint prompt-tuning through CLIP's text and
    vision encoders, which isn't a live-demo-friendly operation. Use the
    report figure/slide for that one.

Run with:
    pip install gradio transformers torch torchvision ftfy
    pip install git+https://github.com/openai/CLIP.git
    python gradio_demo.py
"""

import gradio as gr
import torch
import torch.nn as nn
import clip
import numpy as np
import random
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
from transformers import BlipForImageTextRetrieval
from transformers import AutoImageProcessor, AutoModel
import torch.nn.functional as F
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# -- Device --------------------------------------------------------------
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Loading models on {DEVICE}...')

# -- Load CLIP -------------------------------------------------------------
print('Loading CLIP ViT-B/32...')
clip_model, clip_preprocess = clip.load('ViT-B/32', device=DEVICE)
clip_model.eval()
print('CLIP loaded.')

# -- Load BLIP -------------------------------------------------------------
print('Loading BLIP captioning model...')
blip_processor = BlipProcessor.from_pretrained('Salesforce/blip-image-captioning-base')
blip_caption_model = BlipForConditionalGeneration.from_pretrained(
    'Salesforce/blip-image-captioning-base',
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32
).to(DEVICE)
blip_caption_model.eval()

print('Loading BLIP ITM model...')
blip_itm_model = BlipForImageTextRetrieval.from_pretrained(
    'Salesforce/blip-itm-base-coco',
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32
).to(DEVICE)
blip_itm_model.eval()

# -- Load DINOv2 (frozen backbone, public weights) --------------------------
print('Loading DINOv2-small...')
dinov2_processor = AutoImageProcessor.from_pretrained('facebook/dinov2-small')
dinov2_model = AutoModel.from_pretrained('facebook/dinov2-small').to(DEVICE)
dinov2_model.eval()

print('All models loaded. Starting Gradio...')

# -- Shared embedding helpers ------------------------------------------------
def encode_clip(image):
    """Live CLIP image embedding, L2-normalised."""
    img_input = clip_preprocess(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        feat = clip_model.encode_image(img_input)
        feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat.float().cpu()

def encode_dinov2(image):
    """Live DINOv2 image embedding (CLS token), L2-normalised."""
    inputs = dinov2_processor(images=image, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = dinov2_model(**inputs)
        feat = out.last_hidden_state[:, 0, :]
        feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat.float().cpu()

Loading models on cuda...
Loading CLIP ViT-B/32...


100%|███████████████████████████████████████| 338M/338M [00:04<00:00, 86.1MiB/s]


CLIP loaded.
Loading BLIP captioning model...


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

All models loaded. Starting Gradio...


In [3]:
# =============================================================================
# TAB: CLIP Zero-Shot Classification
# =============================================================================

def clip_zero_shot(image, class_names_text, prompt_template):
    if image is None:
        return "Please upload an image."

    class_names = [c.strip() for c in class_names_text.split(',') if c.strip()]
    if len(class_names) < 2:
        return "Please enter at least 2 class names separated by commas."

    prompts = [prompt_template.replace('{}', name) for name in class_names]
    img_input = clip_preprocess(image).unsqueeze(0).to(DEVICE)
    text_tokens = clip.tokenize(prompts).to(DEVICE)

    with torch.no_grad():
        image_features = clip_model.encode_image(img_input)
        text_features  = clip_model.encode_text(text_tokens)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        text_features  = text_features  / text_features.norm(dim=-1, keepdim=True)
        logits = (100.0 * image_features @ text_features.T).softmax(dim=-1)
        scores = logits[0].cpu().numpy()

    pred_idx   = scores.argmax()
    pred_class = class_names[pred_idx]
    confidence = scores[pred_idx] * 100

    result_text = f"**Prediction: {pred_class}** ({confidence:.1f}% confidence)\n\n**All scores:**\n"
    for name, score in sorted(zip(class_names, scores), key=lambda x: -x[1]):
        bar = '\u2588' * int(score * 30)
        result_text += f"  {name:20s} {score*100:5.1f}%  {bar}\n"
    result_text += "\n**No training was used. This is pure zero-shot classification.**"
    return result_text

In [4]:
# =============================================================================
# TAB: Few-Shot Prototype Classification (CLIP)
# =============================================================================

support_store = {}

def add_support_image(image, class_name):
    if image is None:
        return "Please upload an image.", format_support_status()
    if not class_name.strip():
        return "Please enter a class name.", format_support_status()
    cls = class_name.strip().lower()
    feat = encode_clip(image)
    support_store.setdefault(cls, []).append(feat)
    return f"\u2705 Added image #{len(support_store[cls])} to class '{cls}'.", format_support_status()

def format_support_status():
    if not support_store:
        return "Support set is empty. Add some images above."
    lines = ["**Current support set:**"]
    for cls, feats in support_store.items():
        lines.append(f"  - {cls}: {len(feats)} image(s)")
    return "\n".join(lines)

def clear_support():
    support_store.clear()
    return "Support set cleared.", format_support_status()

def few_shot_predict(query_image):
    if query_image is None:
        return "Please upload a query image."
    if len(support_store) < 2:
        return "Please add images for at least 2 classes to the support set first."

    prototypes = {}
    for cls, feats in support_store.items():
        stacked = torch.cat(feats, dim=0)
        proto = stacked.mean(dim=0, keepdim=True)
        prototypes[cls] = proto / proto.norm(dim=-1, keepdim=True)

    query_feat = encode_clip(query_image)
    class_names = list(prototypes.keys())
    proto_matrix = torch.cat([prototypes[c] for c in class_names], dim=0)
    sims = (query_feat @ proto_matrix.T)[0]
    probs = F.softmax(sims * 20, dim=0).numpy()

    pred_idx, pred_class, confidence = probs.argmax(), class_names[probs.argmax()], probs.max() * 100
    result = f"**Prediction: {pred_class}** ({confidence:.1f}% confidence)\n\n**Similarity to each class prototype:**\n"
    for name, prob in sorted(zip(class_names, probs), key=lambda x: -x[1]):
        bar = '\u2588' * int(prob * 30)
        result += f"  {name:20s} {prob*100:5.1f}%  {bar}\n"
    return result

In [5]:
# =============================================================================
# TAB: BLIP Image Captioning
# =============================================================================

def blip_caption(image, max_tokens, num_beams):
    if image is None:
        return "Please upload an image."
    inputs = blip_processor(images=image, return_tensors='pt').to(
        DEVICE, torch.float16 if DEVICE == 'cuda' else torch.float32)
    with torch.no_grad():
        output_ids = blip_caption_model.generate(
            **inputs, max_new_tokens=int(max_tokens), num_beams=int(num_beams), early_stopping=True)
    caption = blip_processor.decode(output_ids[0], skip_special_tokens=True)
    result  = f"**Generated Caption:**\n\n> {caption}\n\n"
    result += f"**Settings:** max_tokens={int(max_tokens)}, beam_search_width={int(num_beams)}\n\n"
    result += "**No task-specific training -- this is zero-shot captioning from BLIP's pre-training.**"
    return result

In [6]:
# =============================================================================
# TAB: BLIP Image-Text Matching
# =============================================================================

def blip_match(image, text_descriptions):
    if image is None:
        return "Please upload an image."
    descriptions = [d.strip() for d in text_descriptions.strip().split('\n') if d.strip()]
    if not descriptions:
        return "Please enter at least one text description."

    results = []
    for desc in descriptions:
        inputs = blip_processor(images=image, text=desc, return_tensors='pt').to(
            DEVICE, torch.float16 if DEVICE == 'cuda' else torch.float32)
        with torch.no_grad():
            output = blip_itm_model(**inputs, use_itm_head=True)
            match_prob = torch.nn.functional.softmax(output.itm_score, dim=1)[0][1].item()
        results.append((desc, match_prob))
    results.sort(key=lambda x: -x[1])

    output = "**Image-Text Match Scores** (higher = better match):\n\n"
    for desc, prob in results:
        bar   = '\u2588' * int(prob * 40)
        emoji = '\u2705' if prob > 0.7 else '\u26a0\ufe0f' if prob > 0.4 else '\u274c'
        output += f"{emoji} **{prob*100:.1f}%** -- {desc}\n    {bar}\n\n"
    return output

In [7]:
# =============================================================================
# LIVE ADAPTER METHODS -- run on your support set, no checkpoints needed
# =============================================================================
# Tip-Adapter: Zhang et al., 2022 -- training-free cache-based adapter.
#   logits = alpha * (support-cache affinity) + zero-shot CLIP logits
# CLIP-LoRA (live probe): a small linear layer on top of frozen CLIP features,
#   trained for a few steps of gradient descent on the support set. This is a
#   live stand-in for "fast few-shot adaptation", not the paper's low-rank
#   attention adapters (those need backprop through CLIP itself).
# 2SFS: prototype stage-1 (as in the Few-Shot tab) followed by a stage-2
#   linear calibration head, also trained live in a few steps.

def tip_adapter_predict(query_feat, class_names, feats_by_class, zero_shot_logits=None, alpha=1.0, beta=5.5):
    """Training-free cache adapter: blend affinity to cached support features
    with (optional) CLIP zero-shot text logits. No gradient steps at all."""
    cache_keys, cache_labels = [], []
    for ci, cls in enumerate(class_names):
        for f in feats_by_class[cls]:
            cache_keys.append(f)
            cache_labels.append(ci)
    cache_keys = torch.cat(cache_keys, dim=0)                      # [N, D]
    cache_labels_1hot = F.one_hot(torch.tensor(cache_labels), num_classes=len(class_names)).float()

    affinity = query_feat @ cache_keys.T                           # [1, N]
    cache_logits = ((-1) * (beta - beta * affinity)).exp() @ cache_labels_1hot
    logits = alpha * cache_logits
    if zero_shot_logits is not None:
        logits = logits + zero_shot_logits
    return F.softmax(logits, dim=-1)[0].numpy()

def train_linear_probe(feats_by_class, class_names, in_dim, steps=60, lr=0.05):
    """Tiny live logistic-regression head on frozen features -- the
    'CLIP-LoRA (live probe)' and stage-2 of '2SFS'."""
    X, y = [], []
    for ci, cls in enumerate(class_names):
        for f in feats_by_class[cls]:
            X.append(f.squeeze(0))
            y.append(ci)
    X = torch.stack(X, dim=0)
    y = torch.tensor(y)
    head = nn.Linear(in_dim, len(class_names))
    opt = torch.optim.Adam(head.parameters(), lr=lr)
    for _ in range(steps):
        opt.zero_grad()
        loss = F.cross_entropy(head(X), y)
        loss.backward()
        opt.step()
    head.eval()
    return head

def two_stage_fs_predict(query_feat, prototypes, class_names, head):
    """Stage 1: nearest-prototype score. Stage 2: linear head on the same
    frozen features. Final score = average of the two (simple, transparent
    late fusion -- easy to explain live)."""
    proto_matrix = torch.cat([prototypes[c] for c in class_names], dim=0)
    stage1 = F.softmax((query_feat @ proto_matrix.T)[0] * 20, dim=-1)
    with torch.no_grad():
        stage2 = F.softmax(head(query_feat)[0], dim=-1)
    return (0.5 * stage1 + 0.5 * stage2).numpy()

In [8]:
# =============================================================================
# TAB: Live K-Shot Accuracy Lab -- CLIP, DINOv2, Tip-Adapter, CLIP-LoRA
# (live probe), and 2SFS, all computed live on YOUR uploaded images
# =============================================================================

lab_store = {}   # cls -> list of dicts {"clip": feat, "dino": feat}

METHOD_COLORS = {
    'CLIP prototype':        '#2196F3',
    'DINOv2 prototype':      '#F44336',
    'Tip-Adapter (live)':    '#4CAF50',
    'CLIP-LoRA (live probe)':'#9C27B0',
    '2SFS (live)':           '#FF9800',
}

def add_lab_image(image, class_name):
    if image is None:
        return "Please upload an image.", format_lab_status()
    if not class_name.strip():
        return "Please enter a class name.", format_lab_status()
    cls = class_name.strip().lower()
    lab_store.setdefault(cls, []).append({
        "clip": encode_clip(image),
        "dino": encode_dinov2(image),
    })
    return f"\u2705 Added image #{len(lab_store[cls])} to class '{cls}'.", format_lab_status()

def format_lab_status():
    if not lab_store:
        return "No images yet. Add at least 2 classes with 4+ images each to run the curve."
    lines = ["**Live Lab pool:**"]
    for cls, feats in lab_store.items():
        lines.append(f"  - {cls}: {len(feats)} image(s)")
    return "\n".join(lines)

def clear_lab():
    lab_store.clear()
    return "Live Lab pool cleared.", format_lab_status()

def _split(classes, k):
    """One random K-shot support/held-out split, returns per-modality
    support features and a held-out list of (clip_feat, dino_feat, cls)."""
    sup_clip, sup_dino, held = {}, {}, []
    for cls in classes:
        feats = lab_store[cls]
        idxs = list(range(len(feats)))
        random.shuffle(idxs)
        sup_idxs, held_idxs = idxs[:k], idxs[k:]
        sup_clip[cls] = torch.cat([feats[i]["clip"] for i in sup_idxs], dim=0)
        sup_dino[cls] = torch.cat([feats[i]["dino"] for i in sup_idxs], dim=0)
        for i in held_idxs:
            held.append((feats[i]["clip"], feats[i]["dino"], cls))
    return sup_clip, sup_dino, held

def _proto(stack):
    p = stack.mean(dim=0, keepdim=True)
    return p / p.norm(dim=-1, keepdim=True)

def run_live_kshot_curve(n_trials, methods_to_run):
    n_trials = int(n_trials)
    classes = list(lab_store.keys())
    if len(classes) < 2:
        return None, "Add images for at least 2 classes first (Live Lab pool, left)."
    counts = [len(lab_store[c]) for c in classes]
    if min(counts) < 3:
        return None, f"Every class needs at least 3 images (currently: {dict(zip(classes, counts))})."

    max_k = min(counts) - 1
    ks = list(range(1, max_k + 1))
    curves = {m: {"mean": [], "std": []} for m in methods_to_run}

    for k in ks:
        trial_accs = {m: [] for m in methods_to_run}
        for _ in range(n_trials):
            sup_clip, sup_dino, held = _split(classes, k)
            clip_protos = {c: _proto(sup_clip[c]) for c in classes}
            dino_protos = {c: _proto(sup_dino[c]) for c in classes}

            feats_clip_by_class = {c: [sup_clip[c][i:i+1] for i in range(sup_clip[c].shape[0])] for c in classes}
            lora_head, twosfs_head = None, None
            if 'CLIP-LoRA (live probe)' in methods_to_run or '2SFS (live)' in methods_to_run:
                steps = 40 if k <= 3 else 80  # a few more steps once there's more data
                head = train_linear_probe(feats_clip_by_class, classes, in_dim=sup_clip[classes[0]].shape[1], steps=steps)
                lora_head = head
                twosfs_head = head

            correct = {m: 0 for m in methods_to_run}
            for clip_feat, dino_feat, true_cls in held:
                if 'CLIP prototype' in methods_to_run:
                    pm = torch.cat([clip_protos[c] for c in classes], dim=0)
                    pred = classes[int((clip_feat @ pm.T)[0].argmax())]
                    correct['CLIP prototype'] += int(pred == true_cls)
                if 'DINOv2 prototype' in methods_to_run:
                    pm = torch.cat([dino_protos[c] for c in classes], dim=0)
                    pred = classes[int((dino_feat @ pm.T)[0].argmax())]
                    correct['DINOv2 prototype'] += int(pred == true_cls)
                if 'Tip-Adapter (live)' in methods_to_run:
                    zs = torch.cat([clip_protos[c] for c in classes], dim=0)
                    zero_shot_logits = (clip_feat @ zs.T) * 20
                    probs = tip_adapter_predict(clip_feat, classes, feats_clip_by_class, zero_shot_logits=zero_shot_logits)
                    pred = classes[int(np.argmax(probs))]
                    correct['Tip-Adapter (live)'] += int(pred == true_cls)
                if 'CLIP-LoRA (live probe)' in methods_to_run:
                    with torch.no_grad():
                        probs = F.softmax(lora_head(clip_feat)[0], dim=-1).numpy()
                    pred = classes[int(np.argmax(probs))]
                    correct['CLIP-LoRA (live probe)'] += int(pred == true_cls)
                if '2SFS (live)' in methods_to_run:
                    probs = two_stage_fs_predict(clip_feat, clip_protos, classes, twosfs_head)
                    pred = classes[int(np.argmax(probs))]
                    correct['2SFS (live)'] += int(pred == true_cls)

            for m in methods_to_run:
                trial_accs[m].append(100.0 * correct[m] / len(held))

        for m in methods_to_run:
            curves[m]["mean"].append(float(np.mean(trial_accs[m])))
            curves[m]["std"].append(float(np.std(trial_accs[m])))

    fig, ax = plt.subplots(figsize=(8.5, 5))
    for m in methods_to_run:
        mean_arr = np.array(curves[m]["mean"])
        std_arr  = np.array(curves[m]["std"])
        color = METHOD_COLORS.get(m, None)
        ax.plot(ks, mean_arr, 'o-', linewidth=2.2, markersize=7, label=m, color=color)
        ax.fill_between(ks, mean_arr - std_arr, mean_arr + std_arr, alpha=0.15, color=color)
    ax.set_xlabel('K (support shots per class)')
    ax.set_ylabel('Held-out accuracy (%)')
    ax.set_ylim(0, 105)
    ax.set_xticks(ks)
    ax.set_title(f'Live K-shot curve -- {len(classes)} classes, computed just now on your images')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    lines = [f"**Classes:** {', '.join(classes)}  |  **Trials/K:** {n_trials} random draws\n"]
    for m in methods_to_run:
        mean_arr = np.array(curves[m]["mean"])
        best_i = int(np.argmax(mean_arr))
        lines.append(f"- **{m}**: best at K={ks[best_i]} -> {mean_arr[best_i]:.1f}%")
    lines.append("\n**Every point above is a live computation on your uploaded images.** "
                 "Tip-Adapter and the prototypes are training-free (pure forward pass + closed "
                 "form). The CLIP-LoRA and 2SFS curves involve a few dozen gradient steps on a "
                 "small linear head over frozen CLIP features -- a live approximation of fast "
                 "few-shot adaptation, not the report's exact low-rank-adapter setup (that needs "
                 "backprop through CLIP itself and isn't practical to run live in a talk). "
                 "MaPLe is intentionally left out for the same reason -- see the slides for that figure.")

    return fig, "\n".join(lines)

In [9]:
# =============================================================================
# AUTO-FETCH EXAMPLE IMAGES BY CLASS NAME -- live web image search
# =============================================================================
# Instead of manually uploading images for each class, type class names and
# this searches the live web and downloads a handful of results per class,
# feeding them straight into the Live K-Shot Lab pool.
#
# DuckDuckGo's image search will happily rate-limit you (HTTP 403) after just
# a few back-to-back queries -- that's DDG throttling the endpoint, not a bug
# here. To cope: each class query retries with backoff, classes are queried
# with a short delay between them, and if DDG still refuses, this falls back
# to scraping Bing Images instead. Best-effort throughout: some downloads
# will fail or be a bad match (watermarked, wrong subject) -- glance at your
# pool before presenting.

import re
import time
import random
import requests
from io import BytesIO
try:
    from duckduckgo_search import DDGS
except ImportError:
    from ddgs import DDGS  # package was renamed to `ddgs` in newer releases

_UA = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

def _download(url, timeout=6):
    resp = requests.get(url, timeout=timeout, headers=_UA)
    resp.raise_for_status()
    return Image.open(BytesIO(resp.content)).convert("RGB")

def _fetch_urls_ddg(query, n, max_retries=3):
    """Try DuckDuckGo image search with exponential backoff on rate limits."""
    last_err = None
    for attempt in range(max_retries):
        try:
            with DDGS() as ddgs:
                results = list(ddgs.images(keywords=query, max_results=n * 3))
            return [r.get("image") for r in results if r.get("image")], None
        except Exception as e:
            last_err = e
            time.sleep((2 ** attempt) + random.uniform(0, 1))  # backoff: ~1s, 2s, 4s
    return [], f"DuckDuckGo failed after {max_retries} tries ({last_err})"

def _fetch_urls_bing(query, n, timeout=6):
    """Fallback: scrape Bing Images search results (no API key)."""
    url = f"https://www.bing.com/images/search?q={requests.utils.quote(query)}&form=HDRSC2"
    try:
        resp = requests.get(url, headers=_UA, timeout=timeout)
        resp.raise_for_status()
    except Exception as e:
        return [], f"Bing search failed ({e})"
    urls = re.findall(r'murl&quot;:&quot;(.*?)&quot;', resp.text)
    if not urls:
        return [], "Bing returned no results"
    return urls[: n * 3], None

def fetch_images_for_class(query, n=6, timeout=6):
    """Search the web for up to n images matching `query`, return as PIL Images.
    Tries DuckDuckGo first (with retries), falls back to Bing if DDG keeps
    rate-limiting. Over-fetches candidate URLs since some will fail to
    download."""
    urls, err = _fetch_urls_ddg(query, n)
    backend = "DuckDuckGo"
    if not urls:
        urls, err2 = _fetch_urls_bing(query, n, timeout=timeout)
        backend = "Bing (fallback)"
        if not urls:
            return [], f"{err}; {err2}"

    images = []
    for url in urls:
        if len(images) >= n:
            break
        try:
            images.append(_download(url, timeout=timeout))
        except Exception:
            continue  # skip broken/blocked/non-image URLs, keep going

    if not images:
        return [], f"found candidate URLs via {backend} but none downloaded successfully"
    return images, None

def auto_populate_lab(class_names_text, images_per_class):
    """Fetch images for each comma-separated class name and add them straight
    to the Live K-Shot Lab pool (lab_store), encoding with both CLIP and
    DINOv2 as they come in. Classes are queried one at a time with a short
    delay between them to avoid tripping search-engine rate limits."""
    class_names_text = (class_names_text or "").strip()
    if not class_names_text:
        return "Please enter at least one class name.", format_lab_status()

    classes = [c.strip() for c in class_names_text.split(',') if c.strip()]
    n = int(images_per_class)
    log_lines = []
    for i, cls in enumerate(classes):
        if i > 0:
            time.sleep(2.5 + random.uniform(0, 1.5))  # space out requests across classes
        images, err = fetch_images_for_class(cls, n=n)
        if err and not images:
            log_lines.append(f"\u26a0\ufe0f **{cls}**: {err}")
            continue
        for img in images:
            lab_store.setdefault(cls.lower(), []).append({
                "clip": encode_clip(img),
                "dino": encode_dinov2(img),
            })
        status = "\u2705" if len(images) == n else "\u26a0\ufe0f"
        log_lines.append(f"{status} **{cls}**: fetched {len(images)}/{n} images")

    log_lines.append("")
    log_lines.append("_Fetched live via web image search (DuckDuckGo, with a Bing fallback if "
                     "rate-limited) -- these are real, unreviewed web images. Worth a quick "
                     "visual scan for anything mislabeled or watermarked before you demo. If a "
                     "class keeps failing, wait a minute and try again, or add a couple of "
                     "images for it manually below._")
    return "\n".join(log_lines), format_lab_status()

In [10]:
# =============================================================================
# TAB: About / How It Works
# =============================================================================

ABOUT_TEXT = """
# Few-Shot Learning with CLIP, BLIP, and DINOv2
## IUT CSE Semester 6 -- CVLab Design Project

---

Every tab in this demo computes its result live, on whatever image(s) you
upload, using public pretrained backbones -- no lookup tables, no cached
results.

**Training-free tabs** (pure forward pass): CLIP Zero-Shot, Few-Shot
Prototype, BLIP Captioning, BLIP Matching, and (in the K-Shot Lab) the CLIP
and DINOv2 prototype curves and Tip-Adapter.

**Live-trained tabs**: the K-Shot Lab's "CLIP-LoRA (live probe)" and "2SFS"
curves fit a small linear head on your support embeddings in real time (a
few dozen gradient steps, a couple of seconds) -- a live illustration of fast
few-shot adaptation, not a reproduction of the report's exact adapter
architectures or numbers.

**Not reproduced live**: MaPLe, and the report's full five-adapter benchmark
table -- those used trained checkpoints and a fixed dataset not bundled with
this Colab session. See the project slides/report for those figures.
"""

def get_about():
    return ABOUT_TEXT

In [ ]:
# =============================================================================
# BUILD + LAUNCH THE GRADIO APP
# =============================================================================

with gr.Blocks(title="Few-Shot Learning: CLIP + BLIP + DINOv2") as demo:
    gr.Markdown("# Few-Shot Learning with CLIP, BLIP, and DINOv2\nIUT CSE Semester 6 -- CVLab")

    with gr.Tab("CLIP Zero-Shot"):
        gr.Markdown("Classify an image against class names you type in -- no training, no examples.")
        with gr.Row():
            with gr.Column():
                zs_image = gr.Image(type="pil", label="Image")
                zs_classes = gr.Textbox(label="Class names (comma-separated)", value="cat, dog, bird")
                zs_prompt = gr.Textbox(label="Prompt template", value="a photo of a {}")
                zs_btn = gr.Button("Classify", variant="primary")
            zs_output = gr.Markdown()
        zs_btn.click(clip_zero_shot, inputs=[zs_image, zs_classes, zs_prompt], outputs=zs_output)

    with gr.Tab("Few-Shot Prototype"):
        gr.Markdown("Add a few labeled example images per class, then classify a new query image by nearest class prototype (CLIP embeddings).")
        with gr.Row():
            with gr.Column():
                gr.Markdown("**1. Build the support set**")
                sup_image = gr.Image(type="pil", label="Example image")
                sup_class = gr.Textbox(label="Class name")
                sup_add_btn = gr.Button("Add to support set")
                sup_clear_btn = gr.Button("Clear support set")
                sup_add_status = gr.Markdown()
                sup_status = gr.Markdown(format_support_status())
            with gr.Column():
                gr.Markdown("**2. Classify a query image**")
                query_image = gr.Image(type="pil", label="Query image")
                query_btn = gr.Button("Predict", variant="primary")
                query_output = gr.Markdown()
        sup_add_btn.click(add_support_image, inputs=[sup_image, sup_class], outputs=[sup_add_status, sup_status])
        sup_clear_btn.click(clear_support, outputs=[sup_add_status, sup_status])
        query_btn.click(few_shot_predict, inputs=query_image, outputs=query_output)

    with gr.Tab("BLIP Captioning"):
        with gr.Row():
            with gr.Column():
                cap_image = gr.Image(type="pil", label="Image")
                cap_max_tokens = gr.Slider(5, 50, value=20, step=1, label="Max new tokens")
                cap_beams = gr.Slider(1, 8, value=3, step=1, label="Beam search width")
                cap_btn = gr.Button("Generate caption", variant="primary")
            cap_output = gr.Markdown()
        cap_btn.click(blip_caption, inputs=[cap_image, cap_max_tokens, cap_beams], outputs=cap_output)

    with gr.Tab("BLIP Image-Text Matching"):
        gr.Markdown("Enter one candidate description per line -- BLIP scores how well each matches the image.")
        with gr.Row():
            with gr.Column():
                match_image = gr.Image(type="pil", label="Image")
                match_text = gr.Textbox(label="Candidate descriptions (one per line)", lines=5,
                                         value="a dog running on grass\na cat sleeping on a couch\na person riding a bicycle")
                match_btn = gr.Button("Score matches", variant="primary")
            match_output = gr.Markdown()
        match_btn.click(blip_match, inputs=[match_image, match_text], outputs=match_output)

    with gr.Tab("Live K-Shot Lab"):
        gr.Markdown(
            "Upload several images per class (3+ recommended), pick which methods to compare, "
            "then run a live K-shot curve. Every point is computed on your images right now -- "
            "see the About tab for exactly what 'live' means for each method."
        )
        gr.Markdown("**Auto-fetch by class name** (pulls real images live via web image search -- no manual uploading needed)")
        with gr.Row():
            auto_classes = gr.Textbox(label="Class names (comma-separated)", placeholder="chihuahua, pomeranian", scale=3)
            auto_n = gr.Slider(3, 15, value=6, step=1, label="Images per class", scale=2)
            auto_fetch_btn = gr.Button("Auto-fetch images", variant="secondary", scale=1)
        auto_fetch_status = gr.Markdown()
        gr.Markdown("--- \n**...or add images manually**")
        with gr.Row():
            with gr.Column(scale=1):
                lab_image = gr.Image(type="pil", label="Example image")
                lab_class = gr.Textbox(label="Class name")
                lab_add_btn = gr.Button("Add to pool")
                lab_clear_btn = gr.Button("Clear pool")
                lab_add_status = gr.Markdown()
                lab_status = gr.Markdown(format_lab_status())
                lab_trials = gr.Slider(3, 30, value=10, step=1, label="Random trials per K")
                lab_methods = gr.CheckboxGroup(
                    choices=list(METHOD_COLORS.keys()),
                    value=list(METHOD_COLORS.keys()),
                    label="Methods to compare"
                )
                lab_run_btn = gr.Button("Run live K-shot curve", variant="primary")
            with gr.Column(scale=2):
                lab_plot = gr.Plot(label="Live K-shot curve")
                lab_summary = gr.Markdown()
        auto_fetch_btn.click(auto_populate_lab, inputs=[auto_classes, auto_n], outputs=[auto_fetch_status, lab_status])
        lab_add_btn.click(add_lab_image, inputs=[lab_image, lab_class], outputs=[lab_add_status, lab_status])
        lab_clear_btn.click(clear_lab, outputs=[lab_add_status, lab_status])
        lab_run_btn.click(run_live_kshot_curve, inputs=[lab_trials, lab_methods], outputs=[lab_plot, lab_summary])

    with gr.Tab("About"):
        gr.Markdown(get_about())

demo.queue()
demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://06736dde0a785de22a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replac